# Классификация ботов — quickstart

Ноутбук показывает, как загрузить данные, собрать пару простейших признаков и получить
валидный `submission.csv`. Это **не** решение задачи: скор такого baseline будет чуть выше
константы. Дальше — ваша работа.

Условие и описание метрики — в `README.md`.

In [1]:
import numpy as np
import pandas as pd
import re
import math
from collections import Counter

from feature_engineering import calculate_entropy, add_ua_features

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [2]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 328905 entries, 0 to 328904
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   cookie_id      328905 non-null  str           
 1   event_ts       328905 non-null  datetime64[us]
 2   eid            328905 non-null  int64         
 3   event_name     328905 non-null  str           
 4   platform       328905 non-null  str           
 5   user_agent     328905 non-null  str           
 6   item_id        214308 non-null  float64       
 7   item_category  295985 non-null  str           
 8   item_location  305112 non-null  str           
 9   seller_type    195052 non-null  str           
 10  search_query   100402 non-null  str           
 11  search_page    100402 non-null  float64       
 12  pointer_x      108540 non-null  float64       
 13  pointer_y      108540 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(8)
memory usage

In [4]:
events['platform'] = events['platform'].str.lower().astype('category')
events['item_category'] =  events['item_category'].fillna('unknown').astype('category')
events['item_location'] = events['item_location'].fillna('unknown').astype('category')
events['seller_type'] = events['seller_type'].fillna('unknown').astype('category')
events['event_name'] = events['event_name'].astype('category')
q = events['search_query'].fillna('').astype('string')
events['query_entropy']   = q.apply(calculate_entropy)

## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [5]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
web        138792
android    126875
desktop     46866
ios         12269
iphone       4103
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.000
item_location    0.000
seller_type      0.000
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
query_entropy    0.000
dtype: float64


## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [6]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


In [7]:
def extract_all_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    meta = meta.copy()
    meta['cookie_age_days'] = (meta['window_start_ts'] - meta['cookie_created_at']).dt.total_seconds() / 86400.0

    ev = ev.sort_values(['cookie_id', 'event_ts'])
    ev['time_diff'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

    agg_spec = {
        'eid': 'count',
        'item_id': 'nunique',
        'item_category': 'nunique',
        'item_location': 'nunique',
        'search_query': 'nunique',
        'search_page': ['max', 'mean'],
        'entropy': ['mean', 'min', 'max'],
        'is_known_tool': 'max',
        'is_headless': 'max',
        'has_url': 'max',
        'starts_with_mozilla': 'mean',
        'is_app_header': 'max',
        'ua_device_conflict': 'max',
        'ua_length': ['mean', 'std'],
        'digit_ratio': 'mean',
        'chrome_major_version': ['min', 'max', 'nunique'],
        'time_diff': ['mean', 'std', 'min', 'median'],
        'pointer_x': lambda x: x.notna().mean(),
    }

    features = ev.groupby('cookie_id').agg(agg_spec)
    features.columns = ['_'.join(c).strip('_') for c in features.columns]

    event_freq = pd.crosstab(ev['cookie_id'], ev['event_name'], normalize='index')
    event_freq.columns = [f'freq_{c}' for c in event_freq.columns]

    df_out = meta[['cookie_id', 'cookie_age_days']].merge(features.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(event_freq.reset_index(), on='cookie_id', how='left')

    return df_out.fillna(0)

In [8]:
add_ua_features(events)

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

Xtr = extract_all_features(ev_tr, train)
Xte = extract_all_features(ev_te, test)

feature_cols = [c for c in Xtr.columns if c != 'cookie_id']
ytr = train.target.values

print(f"Количество признаков: {len(feature_cols)}")

Количество признаков: 37


## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [9]:
import lightgbm as lgb
from metric import precision_at_recall

is_valid = train.window_start_ts.ge('2026-04-17').values

X_train_fold, y_train_fold = Xtr.loc[~is_valid, feature_cols], ytr[~is_valid]
X_val_fold, y_val_fold = Xtr.loc[is_valid, feature_cols], ytr[is_valid]

model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train_fold, y_train_fold)

val_preds = model.predict_proba(X_val_fold)[:, 1]
print('P@R0.7 на валидации:', round(precision_at_recall(y_val_fold, val_preds), 4))
print('Константа:', round(y_val_fold.mean(), 4))

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000953 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4714
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

## Сабмит

In [10]:
model.fit(Xtr[feature_cols], ytr)
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[feature_cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

[LightGBM] [Info] Number of positive: 899, number of negative: 10192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000995 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4874
[LightGBM] [Info] Number of data points in the train set: 11091, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081057 -> initscore=-2.428075
[LightGBM] [Info] Start training from score -2.428075
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

,cookie_id,score
0,ck_315fb710a0e371e7,0.001530
1,ck_a76ee3b3e3e522fd,0.042675
2,ck_94c9a4d382689e82,0.016399
3,ck_8eaf9509ad9462a0,0.002710
4,ck_9a88a5a989cb5bc6,0.009995


## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.